# SemScore evaluation release copy

This is a cleaned copy of the supplied scoring implementation. It reads precomputed answers; it does not generate answers or establish their model identities or test-split origin. Stored outputs and private Drive paths were removed. The scoring and averaging functions are retained unchanged for auditability.

Set `CAEE_EVALUATION_CSV` to an authorized local comparison CSV. Results are written locally under `CAEE_EVALUATION_OUTPUT_DIR` (default: `evaluation_outputs`). Results may contain the input records and must not be published without a separate privacy review. Read `docs/REPRODUCIBILITY.md` for known issues. The full notebook has not been execution-tested during preparation.


In [ ]:
# Step 1: Install required packages and update them
!pip install -qqq -U pandas==2.2.2 transformers datasets accelerate trl bitsandbytes tqdm huggingface_hub pyarrow==16.1.0 openai sentence-transformers

# Step 2: Import necessary libraries
import gc
import os
import re
import torch
import transformers
import datasets
import shutil
import pandas as pd
import nltk
from tqdm import tqdm
from huggingface_hub import login, HfApi
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig, pipeline, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util

# Define the SemScore class (extracted and adapted from semscore.py)
class SemScore:
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def score(self, ref_texts, hyp_texts, metric="cosine"):
        ref_embeddings = self.model.encode(ref_texts, convert_to_tensor=True)
        hyp_embeddings = self.model.encode(hyp_texts, convert_to_tensor=True)

        if metric == "cosine":
            scores = util.pytorch_cos_sim(ref_embeddings, hyp_embeddings)
        else:
            raise ValueError("Unsupported metric. Please use 'cosine'.")

        # Extract scores from tensor
        return [float(score) for score in scores.diag()]

# Credentials and local paths are supplied by the execution environment.
from pathlib import Path
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
evaluation_csv = os.environ.get("CAEE_EVALUATION_CSV")
if not evaluation_csv:
    raise ValueError("Set CAEE_EVALUATION_CSV to an authorized local comparison CSV.")
evaluation_output_dir = Path(os.environ.get("CAEE_EVALUATION_OUTPUT_DIR", "evaluation_outputs"))
evaluation_output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# Load the comparison dataset
num_rows_selected = 256
results_df = pd.read_csv(evaluation_csv).sample(num_rows_selected, random_state=42)

# Instantiate SemScore
#semscore = SemScore()
# Use a bigger, better-performing model for SemScore evaluation
semscore = SemScore(model_name="sentence-transformers/all-mpnet-base-v2")

# Define function for evaluating using SemScore
def evaluate_with_semscore(results_df, suffixes=['base', '4096', '4096x1-25', '4096x1-50', '4096x1-75']):
    metrics = ['coherence', 'consistency', 'fluency', 'relevance']

    for suffix in suffixes:
        score_column = f'{suffix}_semscore'
        results_df[score_column] = None

        for idx, row in tqdm(results_df.iterrows(), total=len(results_df), desc=f"Evaluating {suffix} with SemScore"):
            expected_answer = row['expected_answer']
            generated_answer = row[f'parsed_answer_{suffix}']

            # Score using SemScore
            score = semscore.score([expected_answer], [generated_answer])[0]
            results_df.at[idx, score_column] = score

    return results_df

# Evaluate using SemScore
results_df = evaluate_with_semscore(results_df)

# Summary of results
summary = []

for suffix in ['base', '4096', '4096x1-25', '4096x1-50', '4096x1-75']:
    score_column = f'{suffix}_semscore'
    avg_score = pd.to_numeric(results_df[score_column], errors='coerce').mean()
    summary.append({
        'model': suffix,
        'average_semscore': avg_score
    })

summary_df = pd.DataFrame(summary)

# Display the comparison summary
print(summary_df)

# Save the updated DataFrame with the final results
results_df.to_csv(evaluation_output_dir / f'inference_comparison_with_semscore_metrics_{num_rows_selected}.csv', index=False)
summary_df.to_csv(evaluation_output_dir / f'comparison_summary_semscore_metrics_{num_rows_selected}.csv', index=False)

# Clean up memory before ending the script
gc.collect()
torch.cuda.empty_cache()